# Eksperyment: Test

Test działania eksperymentów.

In [9]:
# === 1. Konfiguracja ===
datasets = {
    "mnist_gelu": {
        "train_images": "../mnist/train-images.idx3-ubyte",
        "train_labels": "../mnist/train-labels.idx1-ubyte",
        "test_images": "../mnist/t10k-images.idx3-ubyte",
        "test_labels": "../mnist/t10k-labels.idx1-ubyte",
        "task": "multiclass",
        "losses": ["categorical_cross_entropy"],
        # "activation": "gelu",
        "activation": "sigmoid",
        "learning_rate": 0.05,
        "hidden_layers": [64, 64, 64],
        "epochs": [20],
        "momentum": True,
        "adaptive_lr": True,
        "batching": {
            "minibatch_auto": {
                "batch_size": "auto",
                "shuffle": True,
            },
            # "full_gd": {
            #     "batch_size": None,
            #     "shuffle": False,
            # },
        },
        "early_stopping": True,
        "patience": 3,
    },
}


In [ ]:
from mlp.mlp import MLP
from mlp.utils import (
    load_mnist_images,
    load_mnist_labels,
    plot_loss,
    plot_accuracy
)
import numpy as np

for dataset_name, cfg in datasets.items():
    print(f"\n=== Dataset: {dataset_name.upper()} ===")

    X_train = load_mnist_images(cfg["train_images"])
    y_train = load_mnist_labels(cfg["train_labels"])
    X_test = load_mnist_images(cfg["test_images"])
    y_test = load_mnist_labels(cfg["test_labels"])

    n_inputs = X_train.shape[1]
    n_classes = 10

    for batching_name, batching_cfg in cfg["batching"].items():
        print(f"\n--- Batching: {batching_name.upper()} ---")

        for loss_name in cfg["losses"]:
            for epoch_count in cfg["epochs"]:
                print(
                    f"\n>>> Loss={loss_name}, "
                    f"Epochs={epoch_count}, "
                    f"Batching={batching_name}"
                )

                model = MLP(
                    layer_sizes=[n_inputs, *cfg["hidden_layers"], n_classes],
                    task=cfg["task"],
                    activation=cfg["activation"],
                    learning_rate=cfg["learning_rate"],
                    seed=42,
                    loss=loss_name,
                    momentum=cfg.get("momentum", False),
                    adaptive_lr=cfg.get("adaptive_lr", False),
                    lr_decay=0.999
                )

                history, weight_hist, acc_hist = model.fit(
                    X_train[:60000],
                    y_train[:60000],
                    epochs=epoch_count,
                    batch_size=batching_cfg["batch_size"],
                    shuffle=batching_cfg["shuffle"],
                    verbose=True,
                    use_tqdm=True,
                    early_stopping=cfg["early_stopping"],
                    patience=cfg["patience"],
                )

                # --- Evaluation (UNCHANGED) ---
                y_pred = model.predict(X_test)
                acc = np.mean(y_pred.ravel() == y_test.ravel())
                print(f"Test accuracy: {acc:.4f}")

                plot_loss(
                    history,
                    title=f"MNIST Loss | {batching_name} | {epoch_count} ep"
                )
                plot_accuracy(
                    acc_hist,
                    title=f"MNIST Accuracy | {batching_name} | {epoch_count} ep"
                )



=== Dataset: MNIST_GELU ===

--- Batching: MINIBATCH_AUTO ---

>>> Loss=categorical_cross_entropy, Epochs=20, Batching=minibatch_auto
>>> Version 9 (mini-batch + early stopping)...


Training:   5%|▌         | 1/20 [00:01<00:36,  1.92s/it, acc=0.1132, loss=2.3057, lr=0.05]

[epoch     1/20] loss=2.305655 acc=0.1132 lr=0.05


Training:  10%|█         | 2/20 [00:03<00:32,  1.82s/it, acc=0.1148, loss=2.2968, lr=0.04995]

[epoch     2/20] loss=2.296828 acc=0.1148 lr=0.04995


...

[epoch     7/20] loss=2.167896 acc=0.3429 lr=0.0497007


Training:  40%|████      | 8/20 [00:21<00:37,  3.12s/it, acc=0.3992, loss=1.9817, lr=0.049651] 

[epoch     8/20] loss=1.981701 acc=0.3992 lr=0.049651


Training:  45%|████▌     | 9/20 [00:24<00:32,  2.94s/it, acc=0.4602, loss=1.6849, lr=0.0496014]

[epoch     9/20] loss=1.684891 acc=0.4602 lr=0.0496014
